# Demand Forecasting with Censored Demand
## Data Exploration & Demand Recovery on FreshRetailNet-50K

---

**Objective**: This notebook introduces the FreshRetailNet-50K dataset (fresh retail e-commerce) and walks through the critical concept of **censored demand** -- why observed sales do not equal true demand when stockouts occur, and how to recover unobserved demand using statistical methods.

**What you will learn**:
1. The structure and characteristics of the FreshRetailNet-50K dataset
2. Key exploratory patterns in fresh retail sales data
3. The formal definition of censored (right-censored) demand and why it matters
4. Three methods to recover true demand from censored observations

**Dataset**: FreshRetailNet-50K -- daily sales data from ~900 stores across 18 cities, covering ~865 fresh products over 90 days (March--July 2024).

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import norm
import warnings

import os
NB_OUT = os.path.join('..', 'notebook_output')
os.makedirs(NB_OUT, exist_ok=True)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

SEED = 42
np.random.seed(SEED)

print('Libraries loaded successfully.')

---
# 1. Dataset Overview

The **FreshRetailNet-50K** dataset contains daily sales records for fresh products (fruits, vegetables, meat, seafood, etc.) from an e-commerce fresh retail network. Each row represents one store-product-day observation.

**Columns**:
| Column | Description |
|--------|-------------|
| `city_id` | City identifier (18 cities) |
| `store_id` | Store identifier (~898 stores) |
| `management_group_id` | Management group for the store |
| `first_category_id` | Top-level product category |
| `second_category_id` | Mid-level product category |
| `third_category_id` | Fine-grained product category |
| `product_id` | Product identifier (~865 products) |
| `dt` | Date of observation |
| `sale_amount` | Observed sales (kg) -- **this is censored when stockouts occur** |
| `stock_hour6_22_cnt` | Hours of stock availability between 6am-10pm (0-16) |
| `discount` | Discount applied |
| `holiday_flag` | Whether the day is a holiday |
| `activity_flag` | Whether a promotional activity was running |
| `precpt` | Precipitation (mm) |
| `avg_temperature` | Average temperature (Celsius) |
| `avg_humidity` | Average humidity (%) |
| `avg_wind_level` | Average wind level |

### 1.1 Load the Data

In [ ]:
# Load train and eval parquet files
train = pd.read_parquet('../../data/freshretailnet/raw/data/train.parquet')
eval_df = pd.read_parquet('../../data/freshretailnet/raw/data/eval.parquet')

print(f'Train shape: {train.shape}')
print(f'Eval shape:  {eval_df.shape}')
print(f'\nColumns: {list(train.columns)}')

In [ ]:
# Parse dates and inspect date ranges
train['dt'] = pd.to_datetime(train['dt'])
eval_df['dt'] = pd.to_datetime(eval_df['dt'])

print(f'Train date range: {train["dt"].min().date()} to {train["dt"].max().date()}')
print(f'Eval date range:  {eval_df["dt"].min().date()} to {eval_df["dt"].max().date()}')
print(f'\nTrain days: {train["dt"].nunique()}')
print(f'Eval days:  {eval_df["dt"].nunique()}')

**Date Split**:
- **Training set**: 2024-03-28 to 2024-06-25 (90 days)
- **Evaluation set**: 2024-06-26 to 2024-07-02 (7 days)

This is a realistic temporal split -- we train on ~3 months and predict the next week.

### 1.2 Create Store-Product Key & Unique Counts

In [ ]:
# Create a composite store-product key
train['sp'] = train['store_id'] * 10000 + train['product_id']
eval_df['sp'] = eval_df['store_id'] * 10000 + eval_df['product_id']

# Unique counts
print('=== Unique Entity Counts (Train) ===')
print(f'Cities:              {train["city_id"].nunique()}')
print(f'Stores:              {train["store_id"].nunique()}')
print(f'Products:            {train["product_id"].nunique()}')
print(f'Store-Products (SP): {train["sp"].nunique()}')
print(f'\n=== Category Hierarchy ===')
print(f'1st-level categories: {train["first_category_id"].nunique()}')
print(f'2nd-level categories: {train["second_category_id"].nunique()}')
print(f'3rd-level categories: {train["third_category_id"].nunique()}')
print(f'\nHierarchy: {train["first_category_id"].nunique()} -> {train["second_category_id"].nunique()} -> {train["third_category_id"].nunique()}')

The category hierarchy follows a 3-level structure: **32 -> 84 -> 233** categories, going from broad groups (e.g., "Fruits") down to fine-grained subcategories (e.g., "Red Fuji Apples").

With ~50K store-product combinations, this is a high-dimensional forecasting problem.

### 1.3 Sample for Notebook Speed

To keep this notebook interactive (fast cell execution), we sample **3,000 store-product combinations** stratified by stockout rate. This ensures our sample includes a mix of frequently-stocked and frequently-out-of-stock products.

In [ ]:
# Compute stockout rate per store-product
sp_stats = train.groupby('sp').agg(
    stockout_rate=('stock_hour6_22_cnt', lambda x: (x < 16).mean()),
    mean_sales=('sale_amount', 'mean'),
    n_days=('dt', 'nunique')
).reset_index()

# Create stockout rate bins for stratification
sp_stats['so_bin'] = pd.cut(
    sp_stats['stockout_rate'],
    bins=[0, 0.1, 0.3, 0.5, 0.7, 1.0],
    labels=['0-10%', '10-30%', '30-50%', '50-70%', '70-100%'],
    include_lowest=True
)

print('Stockout rate distribution across all SPs:')
print(sp_stats['so_bin'].value_counts().sort_index())
print(f'\nTotal SPs: {len(sp_stats)}')

In [ ]:
# Stratified sample: 3000 SPs proportional to stockout bins
N_SAMPLE = 3000

sampled_sps = (
    sp_stats
    .groupby('so_bin', group_keys=False)
    .apply(lambda g: g.sample(
        n=min(len(g), int(np.ceil(N_SAMPLE * len(g) / len(sp_stats)))),
        random_state=SEED
    ))
)

# Trim to exactly 3000 if slightly over due to ceiling
if len(sampled_sps) > N_SAMPLE:
    sampled_sps = sampled_sps.sample(n=N_SAMPLE, random_state=SEED)

sampled_sp_set = set(sampled_sps['sp'].values)

print(f'Sampled {len(sampled_sp_set)} store-product combinations')
print(f'\nStratification check:')
print(sp_stats[sp_stats['sp'].isin(sampled_sp_set)]['so_bin'].value_counts().sort_index())

In [ ]:
# Filter train to sampled SPs
df = train[train['sp'].isin(sampled_sp_set)].copy()

print(f'Sampled train shape: {df.shape}')
print(f'Original train shape: {train.shape}')
print(f'Reduction: {1 - len(df)/len(train):.1%}')

### 1.4 Memory Optimization

Fresh retail datasets can be large. We downcast numeric columns to reduce memory usage.

In [ ]:
mem_before = df.memory_usage(deep=True).sum() / 1e6

# Downcast int64 -> int32
int_cols = df.select_dtypes(include=['int64']).columns
for col in int_cols:
    df[col] = df[col].astype(np.int32)

# Downcast float64 -> float32
float_cols = df.select_dtypes(include=['float64']).columns
for col in float_cols:
    df[col] = df[col].astype(np.float32)

mem_after = df.memory_usage(deep=True).sum() / 1e6

print(f'Memory before: {mem_before:.1f} MB')
print(f'Memory after:  {mem_after:.1f} MB')
print(f'Saved:         {mem_before - mem_after:.1f} MB ({(1 - mem_after/mem_before):.0%} reduction)')

In [ ]:
df.head(10)

In [ ]:
df.describe()

---
# 2. Exploratory Visualizations

Before diving into the censoring problem, let us understand the basic patterns in this dataset.

### 2.1 Distribution of Sale Amount

Fresh product sales are typically right-skewed: most items sell small quantities on a given day, with occasional spikes from promotions or high-demand events.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
axes[0].hist(df['sale_amount'], bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(df['sale_amount'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean = {df["sale_amount"].mean():.2f} kg')
axes[0].axvline(df['sale_amount'].median(), color='orange', linestyle='--', linewidth=2,
                label=f'Median = {df["sale_amount"].median():.2f} kg')
axes[0].set_xlabel('Sale Amount (kg)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Daily Sale Amount')
axes[0].legend()

# Log-transformed for clarity
positive_sales = df.loc[df['sale_amount'] > 0, 'sale_amount']
axes[1].hist(np.log1p(positive_sales), bins=100, color='darkgreen', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('log(1 + Sale Amount)')
axes[1].set_ylabel('Count')
axes[1].set_title('Log-Transformed Sale Amount (positive sales only)')

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb01_sale_amount_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Sale amount statistics:')
print(f'  Mean:   {df["sale_amount"].mean():.3f} kg')
print(f'  Median: {df["sale_amount"].median():.3f} kg')
print(f'  Std:    {df["sale_amount"].std():.3f} kg')
print(f'  Max:    {df["sale_amount"].max():.1f} kg')
print(f'  Zero-sale rows: {(df["sale_amount"] == 0).sum()} ({(df["sale_amount"] == 0).mean():.1%})')

The distribution is heavily right-skewed with a mean around ~1.0 kg. A large number of rows have very small or zero sales -- some of which are due to **stockouts**, not low demand.

### 2.2 Stock Availability (stock_hour6_22_cnt)

This column measures how many hours (out of the 16 operating hours, 6am-10pm) a product had stock on the shelf. A value of 16 means the product was available all day; a value of 0 means the product was out of stock the entire operating period.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

stock_counts = df['stock_hour6_22_cnt'].value_counts().sort_index()
colors = ['#d32f2f' if x < 16 else '#388e3c' for x in stock_counts.index]

ax.bar(stock_counts.index, stock_counts.values, color=colors, edgecolor='white', alpha=0.85)
ax.set_xlabel('Stock Hours (6am-10pm)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Stock Availability Hours')
ax.set_xticks(range(0, 17))

# Add annotations
pct_full = (df['stock_hour6_22_cnt'] == 16).mean()
pct_zero = (df['stock_hour6_22_cnt'] == 0).mean()
ax.annotate(f'{pct_full:.1%} fully stocked', xy=(16, stock_counts.get(16, 0)),
            xytext=(13, stock_counts.get(16, 0) * 0.9),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=11, color='green')
ax.annotate(f'{pct_zero:.1%} zero stock', xy=(0, stock_counts.get(0, 0)),
            xytext=(3, stock_counts.get(0, 0) * 0.9),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=11, color='red')

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb01_stock_availability_hours.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Stock hours distribution:')
print(f'  Fully stocked (16h): {pct_full:.1%}')
print(f'  Zero stock (0h):     {pct_zero:.1%}')
print(f'  Partial stockout:    {1 - pct_full - pct_zero:.1%}')

The distribution is **bimodal**: most observations are either fully stocked (16 hours) or completely out of stock (0 hours). This bimodality is common in fresh retail -- products tend to either be replenished in time or miss the delivery window entirely.

### 2.3 Stockout Rate

We define a **stockout** as any row where `stock_hour6_22_cnt < 16` -- the product was unavailable for at least part of the operating day.

In [ ]:
df['is_stockout'] = (df['stock_hour6_22_cnt'] < 16).astype(np.int32)

stockout_rate = df['is_stockout'].mean()
print(f'Overall stockout rate: {stockout_rate:.1%}')
print(f'  -> {stockout_rate:.1%} of all store-product-day observations experienced some level of stockout.')
print(f'  -> That is {df["is_stockout"].sum():,} out of {len(df):,} rows.')

# Stockout rate by day of week
df['dow'] = df['dt'].dt.dayofweek
df['dow_name'] = df['dt'].dt.day_name()
dow_stockout = df.groupby('dow_name')['is_stockout'].mean().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
)
print(f'\nStockout rate by day of week:')
for day, rate in dow_stockout.items():
    print(f'  {day:>10s}: {rate:.1%}')

A stockout rate of ~44.3% is substantial. Nearly half of all observations may have **understated true demand**. This is why censored demand correction is critical for fresh retail forecasting.

### 2.4 Average Sales by Day of Week

In [ ]:
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Average observed sales by DOW
dow_sales = df.groupby('dow_name')['sale_amount'].mean().reindex(dow_order)
axes[0].bar(dow_order, dow_sales.values, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_ylabel('Mean Sale Amount (kg)')
axes[0].set_title('Average Daily Sales by Day of Week')
axes[0].tick_params(axis='x', rotation=45)

# Stockout rate by DOW
axes[1].bar(dow_order, dow_stockout.values, color='#d32f2f', edgecolor='white', alpha=0.85)
axes[1].set_ylabel('Stockout Rate')
axes[1].set_title('Stockout Rate by Day of Week')
axes[1].tick_params(axis='x', rotation=45)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb01_sales_stockout_by_dow.png'), dpi=150, bbox_inches='tight')
plt.show()

### 2.5 Weather Feature Distributions

In [ ]:
weather_cols = ['avg_temperature', 'avg_humidity', 'precpt']
weather_labels = ['Avg Temperature (C)', 'Avg Humidity (%)', 'Precipitation (mm)']

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, col, label in zip(axes, weather_cols, weather_labels):
    ax.hist(df[col].dropna(), bins=50, color='teal', edgecolor='white', alpha=0.8)
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.set_title(f'Distribution of {label}')
    ax.axvline(df[col].mean(), color='red', linestyle='--', alpha=0.7,
               label=f'Mean={df[col].mean():.1f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb01_weather_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

print('Weather feature summary:')
print(df[weather_cols].describe().round(1).to_string())

### 2.6 Holiday, Activity, and Discount Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Holiday flag
holiday_counts = df['holiday_flag'].value_counts().sort_index()
axes[0].bar(holiday_counts.index.astype(str), holiday_counts.values,
            color=['steelblue', '#d32f2f'], edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Holiday Flag')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Holiday Flag (holiday rate: {df["holiday_flag"].mean():.1%})')

# Activity flag
activity_counts = df['activity_flag'].value_counts().sort_index()
axes[1].bar(activity_counts.index.astype(str), activity_counts.values,
            color=['steelblue', '#ff8f00'], edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Activity Flag')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Activity Flag (activity rate: {df["activity_flag"].mean():.1%})')

# Discount distribution
axes[2].hist(df['discount'].dropna(), bins=50, color='purple', edgecolor='white', alpha=0.8)
axes[2].set_xlabel('Discount')
axes[2].set_ylabel('Count')
axes[2].set_title(f'Discount Distribution (mean: {df["discount"].mean():.2f})')

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb01_holiday_activity_discount.png'), dpi=150, bbox_inches='tight')
plt.show()

### 2.7 Example Store-Product Time Series

Let us look at 6 randomly selected store-product combinations to see what daily sales patterns look like. **Red-shaded regions** indicate days with stockouts (stock < 16 hours).

In [ ]:
# Select 6 random SPs with enough data and some stockouts
candidate_sps = sp_stats[
    (sp_stats['sp'].isin(sampled_sp_set)) &
    (sp_stats['n_days'] >= 60) &
    (sp_stats['stockout_rate'].between(0.2, 0.7)) &
    (sp_stats['mean_sales'] > 0.3)
]
example_sps = candidate_sps.sample(n=6, random_state=SEED)['sp'].values

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, sp_val in enumerate(example_sps):
    ax = axes[idx]
    sp_data = df[df['sp'] == sp_val].sort_values('dt')
    store = sp_val // 10000
    prod = sp_val % 10000

    # Plot sales
    ax.plot(sp_data['dt'], sp_data['sale_amount'], color='steelblue',
            linewidth=1.2, marker='.', markersize=3, label='Observed Sales')

    # Highlight stockout periods in red
    stockout_mask = sp_data['stock_hour6_22_cnt'] < 16
    for _, row in sp_data[stockout_mask].iterrows():
        ax.axvspan(row['dt'] - pd.Timedelta(hours=12),
                   row['dt'] + pd.Timedelta(hours=12),
                   color='red', alpha=0.15)

    ax.set_title(f'Store {store} / Product {prod}  '
                 f'(stockout rate: {stockout_mask.mean():.0%})', fontsize=10)
    ax.set_ylabel('Sale (kg)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.tick_params(axis='x', rotation=30)

    if idx == 0:
        stockout_patch = mpatches.Patch(color='red', alpha=0.2, label='Stockout period')
        ax.legend(handles=[ax.get_lines()[0], stockout_patch], fontsize=8, loc='upper right')

plt.suptitle('6 Random Store-Product Time Series (90-day sales with stockout periods in red)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb01_example_sp_timeseries.png'), dpi=150, bbox_inches='tight')
plt.show()

Notice how sales often drop to zero or near-zero during red-shaded stockout periods. The question is: **was demand really zero on those days, or was the product simply unavailable?** This is the censored demand problem.

---
# 3. The Censoring Problem (Educational Core)

## 3.1 What Is Censored Demand?

In statistics, **censoring** occurs when the value of an observation is only partially known. In the context of retail demand:

> **Observed sales = min(true demand, available supply)**

When a product is in stock all day (`stock_hour6_22_cnt = 16`), the observed sale amount is a good estimate of true demand. But when the product stocks out during the day (`stock_hour6_22_cnt < 16`), customers who arrived after the stockout could not buy the product. Their demand is **unobserved**.

This is formally known as **right-censoring** (from survival analysis and the Tobit model literature):

$$
Y_i = \begin{cases}
D_i & \text{if } D_i \leq C_i \quad \text{(no stockout: demand fully observed)} \\
C_i & \text{if } D_i > C_i \quad \text{(stockout: demand is censored at capacity } C_i\text{)}
\end{cases}
$$

Where:
- $Y_i$ = observed sales
- $D_i$ = true (latent) demand
- $C_i$ = effective capacity/supply available that day

When `stock_hour6_22_cnt` is less than 16, we know the product ran out. The observed sales $Y_i$ is a **lower bound** on true demand $D_i$, i.e., $Y_i \leq D_i$.

### 3.2 Visual Evidence: Sales vs. Stock Hours

In [ ]:
# Scatter plot: sale_amount vs stock_hour6_22_cnt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subsample for plotting speed
plot_sample = df.sample(n=min(20000, len(df)), random_state=SEED)

# Scatter
axes[0].scatter(plot_sample['stock_hour6_22_cnt'], plot_sample['sale_amount'],
                alpha=0.08, s=8, color='steelblue')
axes[0].set_xlabel('Stock Hours (6am-10pm)')
axes[0].set_ylabel('Sale Amount (kg)')
axes[0].set_title('Sale Amount vs Stock Availability Hours')

# Mean sales by stock hour bucket
stock_bins = df.groupby('stock_hour6_22_cnt')['sale_amount'].mean()
colors_bar = ['#d32f2f' if x < 16 else '#388e3c' for x in stock_bins.index]
axes[1].bar(stock_bins.index, stock_bins.values, color=colors_bar, edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Stock Hours (6am-10pm)')
axes[1].set_ylabel('Mean Sale Amount (kg)')
axes[1].set_title('Mean Sales by Stock Availability Hours')
axes[1].set_xticks(range(0, 17))

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb01_sales_vs_stock_hours.png'), dpi=150, bbox_inches='tight')
plt.show()

# Correlation
corr = df[['sale_amount', 'stock_hour6_22_cnt']].corr().iloc[0, 1]
print(f'Correlation between sale_amount and stock_hour6_22_cnt: {corr:.3f}')
print(f'\nMean sales when fully stocked (16h):  {df.loc[df["stock_hour6_22_cnt"] == 16, "sale_amount"].mean():.3f} kg')
print(f'Mean sales during stockout (<16h):     {df.loc[df["stock_hour6_22_cnt"] < 16, "sale_amount"].mean():.3f} kg')

There is a clear **positive correlation** between stock availability and observed sales. This is not because having more stock *causes* more demand -- it is because **stockouts prevent us from observing the demand that was there**.

Products with only a few hours of stock show much lower average sales, but this is an artifact of censoring, not a reflection of true demand.

### 3.3 Single Store-Product Deep Dive: Stockouts Suppress Observed Sales

In [ ]:
# Pick one SP with moderate stockout rate and good data
demo_sp_candidates = sp_stats[
    (sp_stats['sp'].isin(sampled_sp_set)) &
    (sp_stats['n_days'] >= 80) &
    (sp_stats['stockout_rate'].between(0.3, 0.6)) &
    (sp_stats['mean_sales'] > 0.5)
]
demo_sp = demo_sp_candidates.sample(n=1, random_state=SEED + 7)['sp'].values[0]
demo_data = df[df['sp'] == demo_sp].sort_values('dt').copy()
demo_store = demo_sp // 10000
demo_prod = demo_sp % 10000

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})

# Top: Sales time series
ax1.plot(demo_data['dt'], demo_data['sale_amount'], color='steelblue',
         linewidth=1.5, marker='o', markersize=3, label='Observed Sales')

# Highlight stockout days
stockout_days = demo_data[demo_data['stock_hour6_22_cnt'] < 16]
for _, row in stockout_days.iterrows():
    ax1.axvspan(row['dt'] - pd.Timedelta(hours=12),
                row['dt'] + pd.Timedelta(hours=12),
                color='red', alpha=0.2)

# Mean line for non-censored days
non_censored_mean = demo_data.loc[
    demo_data['stock_hour6_22_cnt'] == 16, 'sale_amount'
].mean()
ax1.axhline(non_censored_mean, color='green', linestyle='--', linewidth=1.5,
            label=f'Mean (non-censored days) = {non_censored_mean:.2f} kg')

stockout_patch = mpatches.Patch(color='red', alpha=0.3, label='Stockout period')
ax1.legend(handles=[ax1.get_lines()[0], ax1.get_lines()[1], stockout_patch],
           fontsize=9, loc='upper right')
ax1.set_ylabel('Sale Amount (kg)')
ax1.set_title(f'Store {demo_store} / Product {demo_prod} -- '
              f'90-Day Sales with Stockout Periods Highlighted',
              fontsize=12)

# Bottom: Stock hours
ax2.bar(demo_data['dt'], demo_data['stock_hour6_22_cnt'],
        color=['#388e3c' if h == 16 else '#d32f2f' for h in demo_data['stock_hour6_22_cnt']],
        width=0.8, alpha=0.7)
ax2.set_ylabel('Stock Hours')
ax2.set_xlabel('Date')
ax2.set_ylim(0, 17)
ax2.axhline(16, color='gray', linestyle=':', alpha=0.5)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
ax2.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb01_single_sp_deep_dive.png'), dpi=150, bbox_inches='tight')
plt.show()

so_rate = (demo_data['stock_hour6_22_cnt'] < 16).mean()
print(f'This store-product has a stockout rate of {so_rate:.0%}')
print(f'Mean sales on fully-stocked days: {non_censored_mean:.2f} kg')
print(f'Mean sales on stockout days:      {demo_data.loc[demo_data["stock_hour6_22_cnt"] < 16, "sale_amount"].mean():.2f} kg')

The green dashed line shows the mean sales on **non-censored** days (when the product was fully stocked). Notice how sales on stockout days (red regions) are systematically below this line. The gap between the two is **lost demand** due to stockouts.

### 3.4 The Vicious Cycle of Naive Forecasting

If we naively train a forecasting model on **observed sales** without correcting for censoring, we trigger a dangerous feedback loop:

```
     Stockout occurs
           |
           v
    Observed sales drop (censored)
           |
           v
    Model trains on low sales -> predicts low demand
           |
           v
    Low demand forecast -> order less inventory
           |
           v
    Under-stocking -> MORE stockouts
           |
           v
    (Cycle repeats, demand estimates keep shrinking)
```

This **vicious cycle** is one of the most common and costly mistakes in retail demand planning. The model systematically under-predicts demand for products that experience stockouts, leading to chronic under-ordering and lost revenue.

**The solution**: Before training any forecasting model, we must **recover** (estimate) the true demand on censored days. This is what we will do in the next section.

---
# 4. Demand Recovery Methods

We now implement three increasingly sophisticated methods to estimate true demand on stockout days.

**Key distinction**:
- On **non-censored** days (`stock_hour6_22_cnt == 16`): observed sales = true demand (no correction needed)
- On **censored** days (`stock_hour6_22_cnt < 16`): observed sales < true demand (correction needed)

### 4.1 Method 1: Proportional (Baseline) Recovery

The simplest approach: assume demand is uniformly distributed across the 16 operating hours. If the product was only available for $h$ hours out of 16, then the true demand is:

$$
\hat{D}_\text{proportional} = Y \times \frac{16}{16 - \text{stockout\_hours}} = Y \times \frac{16}{h}
$$

where $h$ = `stock_hour6_22_cnt` and $Y$ = observed `sale_amount`.

**Limitation**: This assumes demand is perfectly uniform across all hours of the day, which is rarely true. Morning and evening peaks typically see higher demand than midday.

In [ ]:
def proportional_recovery(sale_amount, stock_hours):
    """Proportional (baseline) demand recovery.

    Assumes uniform demand across operating hours.
    recovered = sale_amount * 16 / stock_hours

    If stock_hours == 0, we cannot estimate demand this way (return NaN).
    If stock_hours == 16 (fully stocked), no correction needed.
    """
    result = sale_amount.copy().astype(np.float32)
    mask = (stock_hours > 0) & (stock_hours < 16)
    result[mask] = sale_amount[mask] * 16.0 / stock_hours[mask]
    result[stock_hours == 0] = np.nan  # Cannot recover from zero stock hours
    return result

df['demand_proportional'] = proportional_recovery(
    df['sale_amount'].values,
    df['stock_hour6_22_cnt'].values
)

# Summary
censored = df[df['is_stockout'] == 1]
print('Proportional Recovery Summary (censored days only):')
print(f'  Original mean sales:  {censored["sale_amount"].mean():.3f} kg')
print(f'  Recovered mean demand: {censored["demand_proportional"].mean():.3f} kg')
print(f'  Avg recovery ratio:    {(censored["demand_proportional"] / censored["sale_amount"].replace(0, np.nan)).median():.2f}x')

### 4.2 Method 2: Time-Weighted Recovery

In reality, customer demand is **not uniform across the day**. Fresh retail typically sees:
- **Morning peak** (7am-9am): commuters buying breakfast/lunch items
- **Midday lull** (11am-2pm): lower foot traffic
- **Evening peak** (5pm-7pm): dinner shopping

We define an hourly demand profile and use it to compute a weighted recovery factor. The key assumption is that **stockouts tend to start at the end of the available period** -- the product sells throughout the available hours and then runs out.

In [ ]:
# Hourly demand profile (normalized): hours 6-21 (6am to 9pm)
# Higher weights for morning and evening peaks, lower for midday
HOURLY_PROFILE = np.array([
    0.04,  # 6am
    0.07,  # 7am  (morning peak starts)
    0.09,  # 8am  (morning peak)
    0.08,  # 9am  (morning peak ends)
    0.06,  # 10am
    0.05,  # 11am (midday lull starts)
    0.04,  # 12pm
    0.04,  # 1pm
    0.05,  # 2pm  (midday lull ends)
    0.06,  # 3pm
    0.07,  # 4pm
    0.09,  # 5pm  (evening peak starts)
    0.10,  # 6pm  (evening peak)
    0.08,  # 7pm  (evening peak ends)
    0.05,  # 8pm
    0.03,  # 9pm
], dtype=np.float32)

# Normalize to sum to 1
HOURLY_PROFILE = HOURLY_PROFILE / HOURLY_PROFILE.sum()

# Visualize the profile
fig, ax = plt.subplots(figsize=(12, 4))
hours = [f'{h}:00' for h in range(6, 22)]
ax.bar(hours, HOURLY_PROFILE, color='teal', edgecolor='white', alpha=0.85)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Demand Weight (fraction of daily total)')
ax.set_title('Assumed Hourly Demand Profile for Fresh Retail')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb01_hourly_demand_profile.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Profile sums to: {HOURLY_PROFILE.sum():.4f}')
print(f'Morning peak (7-9am) share: {HOURLY_PROFILE[1:4].sum():.1%}')
print(f'Evening peak (5-7pm) share: {HOURLY_PROFILE[11:14].sum():.1%}')

In [ ]:
def time_weighted_recovery(sale_amount, stock_hours, hourly_profile=HOURLY_PROFILE):
    """Time-weighted demand recovery using an hourly demand profile.

    Assumes stockout starts at the end of the stocked period.
    The fraction of demand captured = cumulative profile weight for the stocked hours.
    recovered = sale_amount / fraction_captured
    """
    result = sale_amount.copy().astype(np.float32)

    # Precompute cumulative demand fraction for each possible stock_hours value
    cum_fractions = np.zeros(17, dtype=np.float32)
    for h in range(1, 17):
        cum_fractions[h] = hourly_profile[:h].sum()

    mask = (stock_hours > 0) & (stock_hours < 16)
    stock_int = stock_hours[mask].astype(int)
    fractions = cum_fractions[stock_int]

    # Avoid division by zero
    fractions = np.maximum(fractions, 0.01)
    result[mask] = sale_amount[mask] / fractions
    result[stock_hours == 0] = np.nan

    return result

df['demand_time_weighted'] = time_weighted_recovery(
    df['sale_amount'].values,
    df['stock_hour6_22_cnt'].values
)

# Summary
censored = df[df['is_stockout'] == 1]
print('Time-Weighted Recovery Summary (censored days only):')
print(f'  Original mean sales:   {censored["sale_amount"].mean():.3f} kg')
print(f'  Recovered mean demand: {censored["demand_time_weighted"].mean():.3f} kg')
print(f'  Avg recovery ratio:    {(censored["demand_time_weighted"] / censored["sale_amount"].replace(0, np.nan)).median():.2f}x')

### 4.3 Method 3: Tobit Correction (Inverse Mills Ratio)

The **Tobit model** (Tobin, 1958) is the classical econometric approach to censored data. It treats observed sales as a censored version of a latent normal demand distribution.

**Setup**: For each store-product, assume true daily demand follows:
$$
D_i \sim N(\mu, \sigma^2)
$$

On censored days (where observed sales $Y_i$ is a lower bound on demand), the **expected true demand given it exceeds the observed sales** is:

$$
E[D | D > S] = \mu + \sigma \cdot \frac{\phi(z)}{1 - \Phi(z)}
$$

where:
- $S$ = observed sales (censoring point)
- $z = \frac{S - \mu}{\sigma}$ (standardized censoring point)
- $\phi(z)$ = standard normal PDF
- $\Phi(z)$ = standard normal CDF
- $\frac{\phi(z)}{1 - \Phi(z)}$ is the **Inverse Mills Ratio** (IMR)

We estimate $\mu$ and $\sigma$ for each store-product from their **non-censored** days (where `stock_hour6_22_cnt == 16`). For store-products with fewer than 5 non-censored observations, we fall back to category-level statistics.

In [ ]:
# Step 1: Compute mu and sigma per SP from non-censored days
non_censored = df[df['stock_hour6_22_cnt'] == 16]

sp_params = non_censored.groupby('sp')['sale_amount'].agg(
    mu='mean',
    sigma='std',
    n_obs='count'
).reset_index()

# Fill NaN sigma (when n_obs=1) with a small value
sp_params['sigma'] = sp_params['sigma'].fillna(0.1)

# For SPs with too few non-censored observations (<5), fall back to category stats
MIN_OBS = 5
reliable = sp_params[sp_params['n_obs'] >= MIN_OBS]
unreliable = sp_params[sp_params['n_obs'] < MIN_OBS]

print(f'SPs with >= {MIN_OBS} non-censored days (reliable):   {len(reliable)}')
print(f'SPs with < {MIN_OBS} non-censored days (fallback):    {len(unreliable)}')
print(f'\nFallback rate: {len(unreliable) / len(sp_params):.1%}')

In [ ]:
# Step 2: Compute category-level fallback stats
# We use third_category_id as the finest fallback level
# non_censored already has third_category_id from df, so we group directly
cat3_params = non_censored.groupby('third_category_id')['sale_amount'].agg(
    cat_mu='mean', cat_sigma='std'
).reset_index()
cat3_params['cat_sigma'] = cat3_params['cat_sigma'].fillna(0.1)

# Also prepare an SP-to-category mapping for later merge
sp_to_cat = df[['sp', 'third_category_id', 'second_category_id', 'first_category_id']].drop_duplicates('sp')

print(f'Category-level fallback parameters computed for {len(cat3_params)} categories.')
print(f'  Mean category mu:    {cat3_params["cat_mu"].mean():.3f}')
print(f'  Mean category sigma: {cat3_params["cat_sigma"].mean():.3f}')

In [ ]:
# Step 3: Merge parameters and compute Tobit-corrected demand
# df already has third_category_id, so we only need to merge sp_params and cat3_params
df_tobit = df.merge(sp_params[['sp', 'mu', 'sigma', 'n_obs']], on='sp', how='left')
df_tobit = df_tobit.merge(cat3_params, on='third_category_id', how='left')

# Use category fallback when n_obs < MIN_OBS
fallback_mask = (df_tobit['n_obs'] < MIN_OBS) | df_tobit['mu'].isna()
df_tobit.loc[fallback_mask, 'mu'] = df_tobit.loc[fallback_mask, 'cat_mu']
df_tobit.loc[fallback_mask, 'sigma'] = df_tobit.loc[fallback_mask, 'cat_sigma']

# Ensure sigma is never zero
df_tobit['sigma'] = df_tobit['sigma'].clip(lower=0.01)

print(f'Fallback applied to {fallback_mask.sum():,} rows ({fallback_mask.mean():.1%})')

In [ ]:
def tobit_recovery(sale_amount, stock_hours, mu, sigma):
    """Tobit (Inverse Mills Ratio) demand recovery.

    For censored observations (stock_hours < 16):
      z = (S - mu) / sigma
      E[D | D > S] = mu + sigma * phi(z) / (1 - Phi(z))

    Non-censored observations are returned as-is.
    """
    result = sale_amount.copy().astype(np.float64)
    censored_mask = (stock_hours < 16) & (stock_hours > 0)

    S = sale_amount[censored_mask].astype(np.float64)
    m = mu[censored_mask].astype(np.float64)
    s = sigma[censored_mask].astype(np.float64)

    z = (S - m) / s

    # Clip z to avoid numerical issues with extreme values
    z = np.clip(z, -5, 5)

    phi_z = norm.pdf(z)
    Phi_z = norm.cdf(z)

    # Inverse Mills Ratio
    imr = phi_z / np.maximum(1 - Phi_z, 1e-10)

    # Expected demand given demand > observed sales
    recovered = m + s * imr

    # Recovered demand should be at least observed sales
    recovered = np.maximum(recovered, S)

    result[censored_mask] = recovered

    # For stock_hours == 0, use mu as best estimate
    zero_stock_mask = stock_hours == 0
    result[zero_stock_mask] = mu[zero_stock_mask].astype(np.float64)

    return result.astype(np.float32)

df['demand_tobit'] = tobit_recovery(
    df_tobit['sale_amount'].values,
    df_tobit['stock_hour6_22_cnt'].values,
    df_tobit['mu'].values,
    df_tobit['sigma'].values
)

# Summary
censored = df[df['is_stockout'] == 1]
print('Tobit (IMR) Recovery Summary (censored days only):')
print(f'  Original mean sales:   {censored["sale_amount"].mean():.3f} kg')
print(f'  Recovered mean demand: {censored["demand_tobit"].mean():.3f} kg')
print(f'  Avg recovery ratio:    {(censored["demand_tobit"] / censored["sale_amount"].replace(0, np.nan)).median():.2f}x')

### 4.4 Comparison of Recovery Methods

In [ ]:
# Pick 4 example SPs for visual comparison
comparison_candidates = sp_stats[
    (sp_stats['sp'].isin(sampled_sp_set)) &
    (sp_stats['n_days'] >= 70) &
    (sp_stats['stockout_rate'].between(0.25, 0.65)) &
    (sp_stats['mean_sales'] > 0.3)
]
comparison_sps = comparison_candidates.sample(n=4, random_state=SEED + 3)['sp'].values

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.ravel()

for idx, sp_val in enumerate(comparison_sps):
    ax = axes[idx]
    sp_data = df[df['sp'] == sp_val].sort_values('dt').copy()
    store = sp_val // 10000
    prod = sp_val % 10000

    # Highlight stockout periods
    stockout_mask = sp_data['stock_hour6_22_cnt'] < 16
    for _, row in sp_data[stockout_mask].iterrows():
        ax.axvspan(row['dt'] - pd.Timedelta(hours=12),
                   row['dt'] + pd.Timedelta(hours=12),
                   color='red', alpha=0.1)

    # Plot all series
    ax.plot(sp_data['dt'], sp_data['sale_amount'], 'o-',
            color='steelblue', markersize=2, linewidth=1, alpha=0.8, label='Observed Sales')
    ax.plot(sp_data['dt'], sp_data['demand_proportional'], 's-',
            color='orange', markersize=2, linewidth=0.8, alpha=0.7, label='Proportional')
    ax.plot(sp_data['dt'], sp_data['demand_time_weighted'], '^-',
            color='green', markersize=2, linewidth=0.8, alpha=0.7, label='Time-Weighted')
    ax.plot(sp_data['dt'], sp_data['demand_tobit'], 'd-',
            color='purple', markersize=2, linewidth=0.8, alpha=0.7, label='Tobit (IMR)')

    ax.set_title(f'Store {store} / Product {prod}  (SO rate: {stockout_mask.mean():.0%})',
                 fontsize=10)
    ax.set_ylabel('Amount (kg)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.tick_params(axis='x', rotation=30)

    if idx == 0:
        ax.legend(fontsize=8, loc='upper right')

plt.suptitle('Demand Recovery Comparison: 4 Store-Products', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb01_demand_recovery_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

### 4.5 Distribution of Recovery Ratios

How much does each method adjust censored observations? Let us look at the distribution of the **recovery ratio** (recovered demand / observed sales) across all censored days.

In [ ]:
# Compute recovery ratios for censored days with positive sales
censored_positive = df[(df['is_stockout'] == 1) & (df['sale_amount'] > 0)].copy()

ratios = pd.DataFrame({
    'Proportional': censored_positive['demand_proportional'] / censored_positive['sale_amount'],
    'Time-Weighted': censored_positive['demand_time_weighted'] / censored_positive['sale_amount'],
    'Tobit (IMR)': censored_positive['demand_tobit'] / censored_positive['sale_amount'],
})

# Clip extreme ratios for visualization
ratios = ratios.clip(upper=10)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
colors = ['orange', 'green', 'purple']

for ax, (name, col), color in zip(axes, ratios.items(), colors):
    valid = col.dropna()
    ax.hist(valid, bins=80, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(valid.median(), color='black', linestyle='--', linewidth=1.5,
               label=f'Median = {valid.median():.2f}x')
    ax.set_xlabel('Recovery Ratio (recovered / observed)')
    ax.set_ylabel('Count')
    ax.set_title(f'{name} Recovery Ratio')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(NB_OUT, 'nb01_recovery_ratios.png'), dpi=150, bbox_inches='tight')
plt.show()

print('Recovery Ratio Summary (censored days, positive sales):')
print(f'{"Method":<20s} {"Median":>10s} {"Mean":>10s} {"Std":>10s}')
print('-' * 52)
for name, col in ratios.items():
    valid = col.dropna()
    print(f'{name:<20s} {valid.median():>10.2f}x {valid.mean():>10.2f}x {valid.std():>10.2f}')

In [ ]:
# Overall comparison: mean demand by method
print('=== Overall Demand Statistics (all days) ===')
print(f'{"":<20s} {"Mean":>10s} {"Median":>10s} {"Std":>10s}')
print('-' * 52)
for name, col in [('Observed Sales', 'sale_amount'),
                   ('Proportional', 'demand_proportional'),
                   ('Time-Weighted', 'demand_time_weighted'),
                   ('Tobit (IMR)', 'demand_tobit')]:
    valid = df[col].dropna()
    print(f'{name:<20s} {valid.mean():>10.3f} {valid.median():>10.3f} {valid.std():>10.3f}')

# Save demand recovery summary
demand_summary = pd.DataFrame({
    'Method': ['Observed Sales', 'Proportional', 'Time-Weighted', 'Tobit (IMR)'],
    'Mean': [df['sale_amount'].mean(), df['demand_proportional'].mean(), df['demand_time_weighted'].mean(), df['demand_tobit'].mean()],
    'Median': [df['sale_amount'].median(), df['demand_proportional'].median(), df['demand_time_weighted'].median(), df['demand_tobit'].median()],
    'Std': [df['sale_amount'].std(), df['demand_proportional'].std(), df['demand_time_weighted'].std(), df['demand_tobit'].std()],
})
demand_summary.to_csv(os.path.join(NB_OUT, 'nb01_demand_recovery_summary.csv'), index=False)
print(f'Saved nb01_demand_recovery_summary.csv to {NB_OUT}')

---
# 5. Key Takeaways

## What We Learned

1. **The FreshRetailNet-50K dataset** contains daily sales for ~865 fresh products across ~898 stores in 18 cities. It exhibits high stockout rates (~44.3%) -- a common challenge in fresh retail where products are perishable and supply chains are tight.

2. **Censored demand is pervasive**: Nearly half of all observations may understate true demand because products ran out of stock before the end of the operating day. The stock availability column (`stock_hour6_22_cnt`) tells us exactly how many hours a product was available.

3. **Naive forecasting on observed sales creates a vicious cycle**: Under-stocking leads to stockouts, which leads to low observed sales, which leads models to predict low demand, which leads to more under-stocking. Breaking this cycle requires correcting for censored demand before training.

4. **Three recovery methods** of increasing sophistication:
   - **Proportional**: Simple and interpretable, but assumes uniform hourly demand. Good as a baseline.
   - **Time-Weighted**: Accounts for within-day demand patterns (morning/evening peaks). More realistic than proportional.
   - **Tobit (Inverse Mills Ratio)**: Statistically principled approach using the truncated normal distribution. Leverages per-SP demand statistics with category-level fallback for sparse data.

## When to Use Which Method

| Method | Best When | Limitations |
|--------|-----------|-------------|
| **Proportional** | Quick baseline; no hourly demand info available | Assumes uniform demand across hours |
| **Time-Weighted** | Hourly demand profile is known or can be estimated | Requires knowledge of within-day patterns; assumes stockout at end of stocked period |
| **Tobit (IMR)** | Sufficient non-censored observations exist; demand is approximately normal | Requires normality assumption; needs fallback for sparse data; computationally heavier |

## Why This Matters for Retail

- **Revenue**: Uncorrected censoring systematically under-estimates demand, leading to chronic under-stocking and lost sales.
- **Waste**: Over-correction can lead to over-ordering perishable goods, increasing waste. The right method balances recovery accuracy.
- **Model Quality**: Demand forecasting models trained on corrected demand produce better predictions, which improve inventory decisions downstream.

In the next notebook, we will use these recovered demand estimates as **training targets** for our forecasting models.